# Notebook 02: Transformacion de Datos
## ETL Optimizacion de Rutas - TransCarga S.A.S.

**Objetivo**: Limpiar, transformar y preparar los datos reales para el modelo

**Tiempo estimado**: 10-15 minutos

In [1]:
# CELDA 1: Configuracion Inicial
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurar paths
BASE_DIR = r'C:\Users\danie\OneDrive\Documentos\TransCarga_ETL'
RAW_DIR = os.path.join(BASE_DIR, 'datos', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datos', 'processed')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')

# Crear directorios
for d in [RAW_DIR, PROCESSED_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOGS_DIR, 'transformacion.log'), mode='a'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print("=" * 70)
print("TRANSFORMACION DE DATOS - TransCarga S.A.S.")
print("=" * 70)
print(f"Directorio raw: {RAW_DIR}")
print(f"Directorio processed: {PROCESSED_DIR}")
logger.info("Environment de transformacion configurado")

2026-05-11 13:09:38,220 - INFO - Environment de transformacion configurado


TRANSFORMACION DE DATOS - TransCarga S.A.S.
Directorio raw: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\raw
Directorio processed: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed


In [2]:
# CELDA 2: Cargar datos raw
def cargar_datos_raw():
    """
    Carga todos los archivos raw
    """
    datos = {}
    
    archivos = [
        ('divipola', 'divipola_raw.csv'),
        ('combustible', 'combustible_raw.csv'),
        ('parque_automotor', 'parque_automotor_raw.csv'),
        ('terminales', 'terminales_raw.csv'),
        ('clientes', 'clientes_raw.csv'),
        ('vehiculos', 'vehiculos_raw.csv'),
        ('trafico', 'trafico_raw.csv'),
        ('peajes', 'peajes_raw.csv')
    ]
    
    print("\nCargando datos raw...")
    
    for nombre, archivo in archivos:
        path = os.path.join(RAW_DIR, archivo)
        if os.path.exists(path):
            datos[nombre] = pd.read_csv(path)
            print(f"  [OK] {nombre}: {len(datos[nombre])} registros")
            logger.info(f"{nombre}: {len(datos[nombre])} registros cargados")
        else:
            print(f"  [SKIP] {nombre}: archivo no encontrado")
            logger.warning(f"{nombre}: archivo no encontrado")
    
    return datos

datos_raw = cargar_datos_raw()
print(f"\nTotal datasets cargados: {len(datos_raw)}")

2026-05-11 13:09:38,258 - INFO - divipola: 1121 registros cargados


2026-05-11 13:09:38,276 - INFO - combustible: 10000 registros cargados


2026-05-11 13:09:38,278 - WARNING - parque_automotor: archivo no encontrado


2026-05-11 13:09:38,280 - INFO - terminales: 192 registros cargados


2026-05-11 13:09:38,282 - INFO - clientes: 200 registros cargados


2026-05-11 13:09:38,283 - INFO - vehiculos: 85 registros cargados


2026-05-11 13:09:38,285 - INFO - trafico: 1176 registros cargados


2026-05-11 13:09:38,287 - INFO - peajes: 6 registros cargados



Cargando datos raw...
  [OK] divipola: 1121 registros
  [OK] combustible: 10000 registros
  [SKIP] parque_automotor: archivo no encontrado
  [OK] terminales: 192 registros
  [OK] clientes: 200 registros
  [OK] vehiculos: 85 registros
  [OK] trafico: 1176 registros
  [OK] peajes: 6 registros

Total datasets cargados: 7


In [3]:
# CELDA 3: Limpiar DIVIPOLA
print("\n" + "=" * 70)
print("1. Limpiando DIVIPOLA")
print("=" * 70)

def limpiar_divipola(df):
    """
    Limpia datos de DIVIPOLA
    """
    df = df.copy()
    
    # Estandarizar nombres de columnas
    df.columns = df.columns.str.lower()
    
    # Eliminar duplicados
    df = df.drop_duplicates()
    
    # Filtrar departamentos de interes
    departamentos_interes = ['antioquia', 'valle del cauca', 'valle_del_cauca', 'valle del cauca']
    df['nom_dpto'] = df['nom_dpto'].str.lower().str.strip()
    
    # Filtrar Antioquia y Valle
    df_antioquia = df[df['nom_dpto'].str.contains('antioquia', na=False)]
    df_valle = df[df['nom_dpto'].str.contains('valle', na=False)]
    df_filtrado = pd.concat([df_antioquia, df_valle])
    
    # Convertir coordenadas
    df_filtrado['latitud'] = pd.to_numeric(df_filtrado['latitud'], errors='coerce')
    df_filtrado['longitud'] = pd.to_numeric(df_filtrado['longitud'], errors='coerce')
    
    # Eliminar nulos en coordenadas
    df_filtrado = df_filtrado.dropna(subset=['latitud', 'longitud'])
    
    print(f"Registros originales: {len(df)}")
    print(f"Registros filtrados (Antioquia + Valle): {len(df_filtrado)}")
    
    logger.info(f"DIVIPOLA limpio: {len(df_filtrado)} registros")
    
    return df_filtrado

if 'divipola' in datos_raw:
    divipola_clean = limpiar_divipola(datos_raw['divipola'])
    output_path = os.path.join(PROCESSED_DIR, 'divipola_clean.csv')
    divipola_clean.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    display(divipola_clean.head())

2026-05-11 13:09:38,298 - INFO - DIVIPOLA limpio: 167 registros



1. Limpiando DIVIPOLA
Registros originales: 1121
Registros filtrados (Antioquia + Valle): 167
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\divipola_clean.csv


,cod_dpto,nom_dpto,cod_mpio,nom_mpio,tipo,latitud,longitud,geo_municipio
0,5,antioquia,5001,MEDELLÍN,Municipio,6.257590,-75.611031,"{'type': 'Point', 'coordinates': [-75.61103107..."
1,5,antioquia,5002,ABEJORRAL,Municipio,5.803728,-75.438474,"{'type': 'Point', 'coordinates': [-75.43847353..."
2,5,antioquia,5004,ABRIAQUÍ,Municipio,6.627569,-76.085978,"{'type': 'Point', 'coordinates': [-76.08597756..."
3,5,antioquia,5021,ALEJANDRÍA,Municipio,6.365534,-75.090597,"{'type': 'Point', 'coordinates': [-75.09059702..."
4,5,antioquia,5030,AMAGÁ,Municipio,6.032922,-75.708003,"{'type': 'Point', 'coordinates': [-75.7080031,..."


In [4]:
# CELDA 4: Limpiar Combustible
print("\n" + "=" * 70)
print("2. Limpiando Precios de Combustible")
print("=" * 70)

def limpiar_combustible(df):
    """
    Limpia datos de combustible
    """
    df = df.copy()
    
    # Estandarizar columnas
    df.columns = df.columns.str.lower()
    
    # Convertir precio a numerico
    df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
    
    # Filtrar precios validos
    df = df[df['precio'] > 0]
    
    # Filtrar departamentos de interes
    df['departamentonombre'] = df['departamentonombre'].str.lower().str.strip()
    df_filtrado = df[df['departamentonombre'].str.contains('antioquia|valle', na=False)]
    
    print(f"Registros originales: {len(df)}")
    print(f"Registros filtrados: {len(df_filtrado)}")
    
    if len(df_filtrado) > 0:
        print(f"\nPrecio promedio por producto:")
        print(df_filtrado.groupby('producto')['precio'].mean().round(0))
    
    logger.info(f"Combustible limpio: {len(df_filtrado)} registros")
    
    return df_filtrado

if 'combustible' in datos_raw:
    combustible_clean = limpiar_combustible(datos_raw['combustible'])
    output_path = os.path.join(PROCESSED_DIR, 'combustible_clean.csv')
    combustible_clean.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    display(combustible_clean.head())

2026-05-11 13:09:38,324 - INFO - Combustible limpio: 1881 registros



2. Limpiando Precios de Combustible
Registros originales: 10000
Registros filtrados: 1881

Precio promedio por producto:
producto
BIODIESEL CORRIENTE              8480.0
BIODIESEL EXTRA                  8451.0
GASOLINA CORRIENTE               8439.0
GASOLINA CORRIENTE OXIGENADA     8992.0
GASOLINA EXTRA                  10250.0
GASOLINA EXTRA OXIGENADA        11467.0
KEROSENE                         9200.0
Name: precio, dtype: float64
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\combustible_clean.csv


,departamentocodigo,departamentonombre,municipiocodigo,municipionombre,agente,bandera,direccion,producto,precio,estado,fecharegistro
5,76,valle del cauca,76109,BUENAVENTURA,CI COMBUSTIBLES DEL MAR SAS,BIOMAX,FLOTANTE APROXIMADAMENTE 100 MTS AL SUROCCIDEN...,BIODIESEL EXTRA,8255,1,2015-01-05T00:00:00.000
6,76,valle del cauca,76109,BUENAVENTURA,CI COMBUSTIBLES DEL MAR SAS,BIOMAX,FLOTANTE APROXIMADAMENTE 100 MTS AL SUROCCIDEN...,GASOLINA CORRIENTE OXIGENADA,8439,1,2015-01-05T00:00:00.000
7,76,valle del cauca,76109,BUENAVENTURA,CI COMBUSTIBLES DEL MAR SAS,BIOMAX,FLOTANTE APROXIMADAMENTE 100 MTS AL SUROCCIDEN...,GASOLINA CORRIENTE,8439,1,2015-01-05T00:00:00.000
24,76,valle del cauca,76109,BUENAVENTURA,ESTACION DE SERVICIO COMBUSTIBLES Y LUBRICANTE...,TEXACO,Carrera. 21 Calle 9 No. 6-186 La Palera,BIODIESEL EXTRA,10300,1,2015-02-10T00:00:00.000
25,76,valle del cauca,76109,BUENAVENTURA,ESTACION DE SERVICIO COMBUSTIBLES Y LUBRICANTE...,TEXACO,Carrera. 21 Calle 9 No. 6-186 La Palera,GASOLINA EXTRA OXIGENADA,8250,1,2015-02-10T00:00:00.000


In [5]:
# CELDA 5: Limpiar Clientes
print("\n" + "=" * 70)
print("3. Limpiando Clientes")
print("=" * 70)

def limpiar_clientes(df):
    """
    Limpia datos de clientes
    """
    df = df.copy()
    
    # Eliminar duplicados
    df = df.drop_duplicates(subset=['id_cliente'])
    
    # Validar coordenadas
    df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
    df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
    
    # Filtrar coordenadas validas (Colombia)
    df = df[
        (df['latitud'].between(-4.5, 12.5)) &
        (df['longitud'].between(-79, -66))
    ]
    
    # Convertir horarios a minutos
    def horario_a_minutos(horario):
        try:
            h, m = map(int, str(horario).split(':'))
            return h * 60 + m
        except:
            return 360
    
    df['apertura_minutos'] = df['horario_apertura'].apply(horario_a_minutos)
    df['cierre_minutos'] = df['horario_cierre'].apply(horario_a_minutos)
    df['ventana_horas'] = (df['cierre_minutos'] - df['apertura_minutos']) / 60
    
    # Categorizar por volumen
    df['categoria_volumen'] = pd.cut(
        df['capacidad_kg'],
        bins=[0, 100, 500, 10000],
        labels=['Pequeno', 'Mediano', 'Grande']
    )
    
    print(f"Clientes limpios: {len(df)}")
    print(f"\nDistribucion por departamento:")
    print(df['departamento'].value_counts())
    
    logger.info(f"Clientes limpios: {len(df)}")
    
    return df

if 'clientes' in datos_raw:
    clientes_clean = limpiar_clientes(datos_raw['clientes'])
    output_path = os.path.join(PROCESSED_DIR, 'clientes_clean.csv')
    clientes_clean.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    display(clientes_clean.head())

2026-05-11 13:09:38,349 - INFO - Clientes limpios: 200



3. Limpiando Clientes
Clientes limpios: 200

Distribucion por departamento:
departamento
ANTIOQUIA          152
VALLE DEL CAUCA     48
Name: count, dtype: int64
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\clientes_clean.csv


,id_cliente,nombre,municipio,departamento,latitud,longitud,capacidad_kg,horario_apertura,horario_cierre,prioridad,frecuencia_entrega,valor_pedido,apertura_minutos,cierre_minutos,ventana_horas,categoria_volumen
0,C00001,Cliente SANTO DOMINGO 1,SANTO DOMINGO,ANTIOQUIA,6.492906,-75.155264,1000,06:00,19:00,3,diario,2.323282e+06,360,1140,13.0,Grande
1,C00002,Cliente VALPARAÍSO 2,VALPARAÍSO,ANTIOQUIA,5.655982,-75.622511,1000,07:00,20:00,3,semanal,9.500336e+05,420,1200,13.0,Grande
2,C00003,Cliente BETULIA 3,BETULIA,ANTIOQUIA,6.190483,-75.952026,1000,09:00,17:00,2,semanal,7.404946e+05,540,1020,8.0,Grande
3,C00004,Cliente ARGELIA 4,ARGELIA,ANTIOQUIA,5.706357,-75.083291,200,09:00,19:00,3,semanal,4.306705e+06,540,1140,10.0,Mediano
4,C00005,Cliente CAICEDONIA 5,CAICEDONIA,VALLE DEL CAUCA,4.295819,-75.854124,500,06:00,20:00,3,semanal,1.290329e+05,360,1200,14.0,Mediano


In [6]:
# CELDA 6: Limpiar Vehiculos
print("\n" + "=" * 70)
print("4. Limpiando Vehiculos")
print("=" * 70)

def limpiar_vehiculos(df):
    """
    Limpia datos de vehiculos
    """
    df = df.copy()
    
    # Eliminar duplicados
    df = df.drop_duplicates(subset=['id_vehiculo'])
    
    # Calcular eficiencia
    df['eficiencia_kg_galon'] = df['capacidad_kg'] / df['consumo_galon_km']
    
    # Antiguedad
    df['antiguedad_anios'] = 2024 - df['anio']
    df['estado_vehiculo'] = df['antiguedad_anios'].apply(
        lambda x: 'Nuevo' if x <= 3 else ('Medio' if x <= 8 else 'Antiguo')
    )
    
    # Disponibilidad numerica
    df['disponible'] = (df['disponibilidad'] == 'disponible').astype(int)
    
    print(f"Vehiculos limpios: {len(df)}")
    print(f"Disponibles: {df['disponible'].sum()}")
    print(f"\nDistribucion por tipo:")
    print(df['tipo'].value_counts())
    
    logger.info(f"Vehiculos limpios: {len(df)}")
    
    return df

if 'vehiculos' in datos_raw:
    vehiculos_clean = limpiar_vehiculos(datos_raw['vehiculos'])
    output_path = os.path.join(PROCESSED_DIR, 'vehiculos_clean.csv')
    vehiculos_clean.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    display(vehiculos_clean.head())

2026-05-11 13:09:38,367 - INFO - Vehiculos limpios: 85



4. Limpiando Vehiculos
Vehiculos limpios: 85
Disponibles: 64

Distribucion por tipo:
tipo
Automovil      27
Camioneta      27
Motocicleta    12
Camion         11
Bus             8
Name: count, dtype: int64
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\vehiculos_clean.csv


,id_vehiculo,tipo,capacidad_kg,consumo_galon_km,velocidad_promedio,bodega_base,anio,costo_hora,disponibilidad,eficiencia_kg_galon,antiguedad_anios,estado_vehiculo,disponible
0,V001,Automovil,200,0.08,40,Medellin,2022,19000,disponible,2500.000000,2,Nuevo,1
1,V002,Motocicleta,50,0.02,45,Medellin,2017,16000,disponible,2500.000000,7,Medio,1
2,V003,Automovil,200,0.08,40,Medellin,2019,19000,disponible,2500.000000,5,Medio,1
3,V004,Camioneta,500,0.12,35,Medellin,2020,25000,disponible,4166.666667,4,Medio,1
4,V005,Camioneta,500,0.12,35,Medellin,2020,25000,disponible,4166.666667,4,Medio,1


In [7]:
# CELDA 7: Limpiar Trafico
print("\n" + "=" * 70)
print("5. Limpiando Trafico")
print("=" * 70)

def limpiar_trafico(df):
    """
    Limpia datos de trafico
    """
    df = df.copy()
    
    # Velocidad en rango valido
    df['velocidad_promedio'] = df['velocidad_promedio'].clip(5, 80)
    
    # Factor de congestion
    df['factor_congestion'] = df['nivel_congestion'] / 100
    
    # Velocidad ajustada
    df['velocidad_ajustada'] = df['velocidad_promedio'] * (1 - df['factor_congestion'] * 0.3)
    
    # Categorizar hora
    def categorizar_hora(hora):
        if 7 <= hora <= 9 or 17 <= hora <= 19:
            return 'Pico'
        elif 10 <= hora <= 16:
            return 'Normal'
        else:
            return 'Bajo'
    
    df['categoria_hora'] = df['hora'].apply(categorizar_hora)
    
    # Es fin de semana
    df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)
    
    print(f"Trafico limpio: {len(df)} registros")
    print(f"\nVelocidad promedio por categoria:")
    print(df.groupby('categoria_hora')['velocidad_promedio'].mean())
    
    logger.info(f"Trafico limpio: {len(df)}")
    
    return df

if 'trafico' in datos_raw:
    trafico_clean = limpiar_trafico(datos_raw['trafico'])
    output_path = os.path.join(PROCESSED_DIR, 'trafico_clean.csv')
    trafico_clean.to_csv(output_path, index=False)
    print(f"Archivo guardado: {output_path}")
    display(trafico_clean.head())

2026-05-11 13:09:38,385 - INFO - Trafico limpio: 1176



5. Limpiando Trafico
Trafico limpio: 1176 registros

Velocidad promedio por categoria:
categoria_hora
Bajo      47.749351
Normal    37.752187
Pico      25.374830
Name: velocidad_promedio, dtype: float64
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\trafico_clean.csv


,zona,dia_semana,hora,velocidad_promedio,flujo_vehicular,nivel_congestion,factor_congestion,velocidad_ajustada,categoria_hora,es_fin_semana
0,Centro,0,0,46.5,223,94.0,0.940,33.38700,Bajo,0
1,Centro,0,1,49.5,740,37.0,0.370,44.00550,Bajo,0
2,Centro,0,2,40.2,749,42.8,0.428,35.03832,Bajo,0
3,Centro,0,3,49.7,774,85.3,0.853,36.98177,Bajo,0
4,Centro,0,4,42.9,369,85.1,0.851,31.94763,Bajo,0


In [8]:
# CELDA 8: Crear matriz de distancias
print("\n" + "=" * 70)
print("6. Creando Matriz de Distancias")
print("=" * 70)

def calcular_matriz_distancias(clientes_df, n_muestra=50):
    """
    Calcula matriz de distancias usando Haversine
    """
    from math import radians, sin, cos, sqrt, atan2
    
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371  # Radio Tierra en km
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        return round(R * c, 2)
    
    # Seleccionar muestra
    n = min(n_muestra, len(clientes_df))
    clientes_sample = clientes_df.head(n)
    
    # Agregar bodegas
    bodegas = pd.DataFrame([
        {'id_cliente': 'BODEGA_MEDELLIN', 'latitud': 6.2518, 'longitud': -75.5636},
        {'id_cliente': 'BODEGA_CALI', 'latitud': 3.4516, 'longitud': -76.5320}
    ])
    
    puntos = pd.concat([
        bodegas[['id_cliente', 'latitud', 'longitud']],
        clientes_sample[['id_cliente', 'latitud', 'longitud']]
    ], ignore_index=True)
    
    # Calcular matriz
    n_puntos = len(puntos)
    matriz = np.zeros((n_puntos, n_puntos))
    
    for i in range(n_puntos):
        for j in range(n_puntos):
            if i != j:
                matriz[i, j] = haversine(
                    puntos.iloc[i]['latitud'], puntos.iloc[i]['longitud'],
                    puntos.iloc[j]['latitud'], puntos.iloc[j]['longitud']
                )
    
    matriz_df = pd.DataFrame(matriz, index=puntos['id_cliente'], columns=puntos['id_cliente'])
    
    output_path = os.path.join(PROCESSED_DIR, 'matriz_distancias.csv')
    matriz_df.to_csv(output_path)
    
    print(f"Matriz creada: {n_puntos}x{n_puntos}")
    print(f"Distancia min: {matriz[matriz>0].min():.2f} km")
    print(f"Distancia max: {matriz.max():.2f} km")
    print(f"Distancia promedio: {matriz[matriz>0].mean():.2f} km")
    print(f"Archivo guardado: {output_path}")
    
    logger.info(f"Matriz distancias: {n_puntos}x{n_puntos}")
    
    return matriz_df

if 'clientes' in datos_raw:
    matriz_dist = calcular_matriz_distancias(datos_raw['clientes'])
    print("\nMatriz de distancias (primeros 10x10):")
    display(matriz_dist.iloc[:10, :10])


6. Creando Matriz de Distancias


2026-05-11 13:09:38,639 - INFO - Matriz distancias: 52x52


Matriz creada: 52x52
Distancia min: 0.68 km
Distancia max: 593.62 km
Distancia promedio: 177.43 km
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\matriz_distancias.csv

Matriz de distancias (primeros 10x10):


id_cliente,BODEGA_MEDELLIN,BODEGA_CALI,C00001,C00002,C00003,C00004,C00005,C00006,C00007,C00008
id_cliente,,,,,,,,,,
BODEGA_MEDELLIN,0.00,329.33,52.49,66.57,43.47,80.62,219.86,182.91,36.51,279.86
BODEGA_CALI,329.33,0.00,370.97,265.04,311.25,297.73,120.28,146.57,330.16,574.88
C00001,52.49,370.97,0.00,106.44,94.26,87.82,256.26,226.28,41.36,274.69
C00002,66.57,265.04,106.44,0.00,69.72,59.93,153.40,119.84,66.94,341.23
C00003,43.47,311.25,94.26,69.72,0.00,110.13,210.96,165.81,77.30,275.12
C00004,80.62,297.73,87.82,59.93,110.13,0.00,178.58,161.34,51.25,356.09
C00005,219.86,120.28,256.26,153.40,210.96,178.58,0.00,56.31,214.93,484.95
C00006,182.91,146.57,226.28,119.84,165.81,161.34,56.31,0.00,186.31,435.72
C00007,36.51,330.16,41.36,66.94,77.30,51.25,214.93,186.31,0.00,304.90


In [9]:
# CELDA 9: Crear matriz de tiempos
print("\n" + "=" * 70)
print("7. Creando Matriz de Tiempos")
print("=" * 70)

def calcular_matriz_tiempos(matriz_distancias, velocidad_base=35):
    """
    Calcula matriz de tiempos en minutos
    """
    factor_trafico = 1.3
    matriz_tiempo = matriz_distancias / velocidad_base * 60 * factor_trafico
    matriz_tiempo = matriz_tiempo.round(1)
    
    output_path = os.path.join(PROCESSED_DIR, 'matriz_tiempos.csv')
    matriz_tiempo.to_csv(output_path)
    
    print(f"Matriz de tiempos creada")
    vals = matriz_tiempo.values
    print(f"Tiempo min: {vals[vals>0].min():.1f} min")
    print(f"Tiempo max: {vals.max():.1f} min")
    print(f"Tiempo promedio: {vals[vals>0].mean():.1f} min")
    print(f"Archivo guardado: {output_path}")
    
    logger.info("Matriz de tiempos creada")
    
    return matriz_tiempo

matriz_tiempos = calcular_matriz_tiempos(matriz_dist)
display(matriz_tiempos.iloc[:10, :10])

2026-05-11 13:09:38,658 - INFO - Matriz de tiempos creada



7. Creando Matriz de Tiempos
Matriz de tiempos creada
Tiempo min: 1.5 min
Tiempo max: 1322.9 min
Tiempo promedio: 395.4 min
Archivo guardado: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed\matriz_tiempos.csv


id_cliente,BODEGA_MEDELLIN,BODEGA_CALI,C00001,C00002,C00003,C00004,C00005,C00006,C00007,C00008
id_cliente,,,,,,,,,,
BODEGA_MEDELLIN,0.0,733.9,117.0,148.4,96.9,179.7,490.0,407.6,81.4,623.7
BODEGA_CALI,733.9,0.0,826.7,590.7,693.6,663.5,268.1,326.6,735.8,1281.2
C00001,117.0,826.7,0.0,237.2,210.1,195.7,571.1,504.3,92.2,612.2
C00002,148.4,590.7,237.2,0.0,155.4,133.6,341.9,267.1,149.2,760.5
C00003,96.9,693.6,210.1,155.4,0.0,245.4,470.1,369.5,172.3,613.1
C00004,179.7,663.5,195.7,133.6,245.4,0.0,398.0,359.6,114.2,793.6
C00005,490.0,268.1,571.1,341.9,470.1,398.0,0.0,125.5,479.0,1080.7
C00006,407.6,326.6,504.3,267.1,369.5,359.6,125.5,0.0,415.2,971.0
C00007,81.4,735.8,92.2,149.2,172.3,114.2,479.0,415.2,0.0,679.5


In [10]:
# CELDA 10: Resumen de Transformacion
print("\n" + "=" * 70)
print("RESUMEN DE TRANSFORMACION")
print("=" * 70)

archivos = [
    ('divipola_clean.csv', 'DIVIPOLA Limpio'),
    ('combustible_clean.csv', 'Combustible Limpio'),
    ('clientes_clean.csv', 'Clientes Limpios'),
    ('vehiculos_clean.csv', 'Vehiculos Limpios'),
    ('trafico_clean.csv', 'Trafico Limpio'),
    ('matriz_distancias.csv', 'Matriz Distancias'),
    ('matriz_tiempos.csv', 'Matriz Tiempos')
]

resumen = []
for archivo, descripcion in archivos:
    path = os.path.join(PROCESSED_DIR, archivo)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0) if 'matriz' in archivo else pd.read_csv(path)
        size = os.path.getsize(path) / 1024
        reg = f"{df.shape[0]}x{df.shape[1]}" if 'matriz' in archivo else len(df)
        resumen.append({
            'Archivo': archivo,
            'Descripcion': descripcion,
            'Registros': reg,
            'Tamano_KB': round(size, 2),
            'Estado': 'OK'
        })
    else:
        resumen.append({
            'Archivo': archivo,
            'Descripcion': descripcion,
            'Registros': 0,
            'Tamano_KB': 0,
            'Estado': 'NO ENCONTRADO'
        })

df_resumen = pd.DataFrame(resumen)
display(df_resumen)

logger.info("Transformacion completada exitosamente")
print("\n[OK] Transformacion completada")


RESUMEN DE TRANSFORMACION


,Archivo,Descripcion,Registros,Tamano_KB,Estado
0,divipola_clean.csv,DIVIPOLA Limpio,167,21.04,OK
1,combustible_clean.csv,Combustible Limpio,1881,281.74,OK
2,clientes_clean.csv,Clientes Limpios,200,30.28,OK
3,vehiculos_clean.csv,Vehiculos Limpios,85,6.59,OK
4,trafico_clean.csv,Trafico Limpio,1176,65.60,OK
5,matriz_distancias.csv,Matriz Distancias,52x52,17.97,OK
6,matriz_tiempos.csv,Matriz Tiempos,52x52,16.30,OK


2026-05-11 13:09:38,764 - INFO - Transformacion completada exitosamente



[OK] Transformacion completada


---
## Datos Transformados

Ejecutar el notebook `03_carga.ipynb` para generar los archivos finales del modelo de optimizacion.